# Sanity check on pipeline

In [1]:
import pandas as pd
from Bio import SeqIO
import matplotlib.pyplot as plt
import seaborn as sns

%cd ../data/CJ5Z9Y_v0

/groups/cress/projects/efactors/jack/insertion_mapping/data/CJ5Z9Y_v0


In [22]:
sample_dir = "CJ5Z9Y_1"
blast_res = pd.read_csv(f"{sample_dir}/blast/RE_blast.tsv", sep="\t", header=None)
blast_res.columns = ['sseqid', 'sstart', 'send', 'sstrand']
read_re_cts = blast_res['sseqid'].value_counts()
reads_to_keep = read_re_cts[read_re_cts == 1].keys()
filtered_out = blast_res.loc[~blast_res["sseqid"].isin(reads_to_keep)].sort_values(by="sseqid")
filtered_out_reads = filtered_out["sseqid"].unique()
filtered_out_reads

array(['5259bd11-f252-4236-881f-200187030dad',
       '9343e77c-b3a6-4eea-aeb9-b53436181f26'], dtype=object)

In [24]:
filtered_out_read_records = []

records = SeqIO.parse("CJ5Z9Y_1/blast/CJ5Z9Y_1_sample_1.fasta", "fasta")
for record in records:
    if record.id in filtered_out_reads:
        filtered_out_read_records.append(record)


with open(f"pipeline_qc/{sample_dir}.multiRE.fasta", "w") as handle:
    SeqIO.write(filtered_out_read_records, handle, "fasta")

Let's try BLASTing the part of the genome immediately upstrean of the insertion 
against all reads, then recycle the RE BLAST results to verify that we're 
picking up all the insertions. 

In [26]:
wk_dir = sample_dir
sequencing_basename = "CJ5Z9Y_1_sample_1"
query_file = "pipeline_qc/target_re_200.fasta"
# !echo "$wk_dir/blast/$sequencing_basename"


!blastn \
-db "$wk_dir/blast/$sequencing_basename" \
-task blastn-short \
-query "$query_file" \
-perc_identity 90 -qcov_hsp_perc 90 \
-outfmt "6 sseqid sstart send sstrand" \
-word_size 4 \
-soft_masking false \
-dust no \
-max_target_seqs 1000000 \
-out "pipeline_qc/$sample_dir/target_site_blast.tsv"

In [28]:
ts_res = pd.read_csv(f"pipeline_qc/{sample_dir}/target_site_blast.tsv", sep="\t", header=None)
ts_res.columns = ['sseqid', 'sstart', 'send', 'sstrand']
ts_res.head()

,sseqid,sstart,send,sstrand
0,13607f52-738f-40bb-995b-5703b8867c3a,1444,1643,plus
1,a048a8c7-1630-4d7b-8367-19f635db0f8d,9859,10058,plus
2,dcbb6b3a-e527-4306-911b-d990879567f8,2071,2270,plus
3,b4d0f81a-5dbf-45f6-8370-5837cacf78e4,6536,6735,plus
4,6820b4ca-8184-441c-8c06-fe318542532a,2595,2794,plus


In [45]:
hits_other_way = ts_res.loc[ts_res["sseqid"].isin(blast_res["sseqid"].unique())]
print(hits_other_way.shape)
print(hits_other_way.loc[hits_other_way['sseqid'].isin(reads_to_keep)].shape)

# blast_res.loc[blast_res["sseqid"].isin(reads_to_keep), "sseqid"].isin(hits_other_way["sseqid"])


(58, 4)
(58, 4)
